# $$ \textbf{Aprendizaje Supervizado}$$
# $$\text{Tarea 4}$$
# $$\text{Daniel Rojo Mata}$$

# **Ejercicio 1: Clasificación de Imágenes con Bolsa de Características (BoVW)**

Considera un conjunto de imágenes de lugares etiquetadas en 15 categorías, dividido en subconjuntos de entrenamiento (100 imágenes por categoría) y prueba. Por cada imagen se extrajeron vectores de características de 128 dimensiones utilizando el algoritmo **SIFT**.

Se debe calcular la representación de bolsa de características de las imágenes mediante el siguiente procedimiento:

### Procedimiento

1. **Extracción de características**  
   Se obtienen puntos de interés y vectores SIFT de cada imagen. Estos ya se encuentran disponibles en archivos `.npy`.

2. **Agrupamiento de características**  
   Se usa el algoritmo **K-medias** para construir un vocabulario visual de tamaño **10,000**. Cada centroide representa una palabra visual.

3. **Construcción de histogramas**  
   Cada imagen se representa como un **histograma de frecuencias** de palabras visuales.

4. **Reducción de dimensionalidad (opcional)**  
   Se puede aplicar **PCA** a los vectores SIFT y/o a las bolsas de características para mejorar la eficiencia.

### Modelado

Se deben entrenar **tres modelos no lineales distintos** (como *Gradient Boosted Trees*, *Random Forests* o *SVMs*) con las representaciones BoVW extraídas, considerando los siguientes escenarios:

- Usando vectores de características originales.  
- Reduciendo dimensiones de los vectores SIFT.  
- Reduciendo dimensiones de las bolsas de características.  
- Reduciendo tanto vectores SIFT como bolsas BoVW.

> Debido al gran número de vectores, se recomienda usar una variante de K-medias con asignación eficiente (por ejemplo, HNSW mediante `hnswlib`).

### Requisitos adicionales

- Reportar **matrices de confusión** y **exactitud** de cada modelo.
- Justificar la elección del número de componentes en la reducción de dimensiones.
- Contestar las siguientes preguntas:
  - ¿Cuáles son las ventajas y desventajas de los modelos entrenados?
  - ¿Qué es una *stop word* en procesamiento de lenguaje natural (PLN) y cómo podría aplicarse esta noción en clasificación de imágenes?


# Se importan las bibliotecas

In [1]:
# Módulos estándar
import os                        # Para manejar rutas de carpetas y archivos del sistema
import numpy as np               # Para operaciones numéricas y manejo de arreglos (vectores/matrices)

# Agrupamiento y búsqueda de vecinos
from sklearn.cluster import MiniBatchKMeans      # Versión eficiente de KMeans, útil para agrupar vectores SIFT en palabras visuales
from sklearn.neighbors import KDTree             # Estructura para búsqueda rápida del vecino más cercano (asignación de palabras visuales)

# Modelos de clasificación supervisada
from sklearn.svm import SVC                      # Support Vector Classifier con kernel RBF (modelo potente para datos complejos)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# Random Forest: ensamble de árboles robusto al ruido
# Gradient Boosting: modelo secuencial que corrige errores del anterior, muy preciso pero más sensible a hiperparámetros

# Validación cruzada y procesamiento de etiquetas
from sklearn.model_selection import cross_val_score     # Para evaluar clasificadores usando k-fold cross-validation
from sklearn.preprocessing import LabelEncoder          # Para convertir etiquetas de clase en enteros

# Reducción de dimensionalidad
from sklearn.decomposition import PCA                   # Para aplicar análisis de componentes principales a vectores SIFT o histogramas BoVW

# Métricas de evaluación
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
# accuracy_score: mide la proporción de aciertos
# confusion_matrix: muestra en qué clases hubo errores
# classification_report: incluye precision, recall y f1-score por clase

# **AWAS**

In [2]:
n_clusters = 10_000 # La tarea pide con 10,000, pero esto hace que tarde muchísimo en ejecutar ciertas celdas

# **Paso 1**: Carga de vectores SIFT

### Objetivo

Preparar los datos para construir la representación de bolsa de características (BoVW), cargando los descriptores SIFT de cada imagen del conjunto de entrenamiento.

### Estructura del dataset

Cada categoría (por ejemplo, `bedroom`) contiene:
- Las imágenes `.jpg` en: imagedb/data_tarea/train/bedroom/

- Los vectores de características SIFT `.npy` en: imagedb/data_tarea/train/bedroom/features/

### Procedimiento

Se implementó una función que:
- Recorre las carpetas de categorías dentro de `imagedb/data_tarea/train/`.
- Accede a la subcarpeta `features/` de cada categoría.
- Carga todos los archivos `.npy`, cada uno correspondiente a una imagen, que contiene una matriz de vectores SIFT (128 dimensiones por punto de interés).
- Almacena:
- Todos los vectores SIFT en una lista.
- Las etiquetas (índice de la clase) correspondientes a cada imagen.
- Los nombres de las categorías.
- Las rutas de cada imagen `.npy`.

In [3]:
def cargar_vectores_sift(directorio_base):
    """
    Carga los vectores SIFT guardados en archivos .npy por categoría desde un directorio base.

    Parámetros:
    ------------
    directorio_base : str
        Ruta base donde están las carpetas de categorías (cada una con subcarpeta 'features/').

    Retorna:
    --------
    vectores_todos : list of np.ndarray
        Lista de matrices SIFT (una por imagen).
    
    etiquetas : list of int
        Índices enteros que representan la clase/categoría de cada imagen.
    
    categorias : list of str
        Nombres de las carpetas (clases) en orden alfabético.
    
    rutas_imagenes : list of str
        Rutas absolutas a cada archivo .npy cargado.
    """

    vectores_todos = []        # Lista para almacenar matrices SIFT por imagen
    etiquetas = []             # Lista de índices de clase correspondientes a cada imagen
    categorias = sorted(os.listdir(directorio_base))  # Obtiene las clases (carpetas) en orden alfabético
    rutas_imagenes = []        # Guarda las rutas completas de cada archivo .npy

    for etiqueta, categoria in enumerate(categorias):
        # Ruta a la subcarpeta de vectores SIFT: ej. train/bedroom/features/
        ruta_features = os.path.join(directorio_base, categoria, "features")
        
        # Verifica que la subcarpeta exista
        if not os.path.isdir(ruta_features):
            continue

        # Recorre los archivos .npy de esa subcarpeta
        for archivo in os.listdir(ruta_features):
            if archivo.endswith(".npy"):
                path = os.path.join(ruta_features, archivo)  # Ruta completa al archivo
                vectores = np.load(path)                    # Carga la matriz SIFT (shape: N×128)
                vectores_todos.append(vectores)             # Guarda la matriz en la lista
                etiquetas.append(etiqueta)                  # Guarda la etiqueta numérica de la clase
                rutas_imagenes.append(path)                 # Guarda la ruta al archivo .npy

    return vectores_todos, etiquetas, categorias, rutas_imagenes


# OJITO: es necesario que la carpeta de descarga esté en la misma carpteta de este archivo
ruta = "./imagedb/data_tarea/train" 
X_train, y_train, categorias, rutas_train = cargar_vectores_sift(ruta)

# **Paso 2**: Construcción del Vocabulario Visual y Asignación con KDTree

### Objetivo

Agrupar todos los vectores SIFT extraídos de las imágenes en un conjunto de palabras visuales mediante K-medias, y asignar cada vector al centroide más cercano utilizando una búsqueda eficiente con `KDTree`.

---

### Procedimiento

1. Se concatenaron todos los vectores SIFT de las imágenes de entrenamiento en una única matriz de tamaño variable `N × 128`, donde `N` representa el número total de descriptores SIFT.

2. Se entrenó un modelo de `MiniBatchKMeans` con 10,000 clústeres (`n_clusters = 10,000`) para construir el vocabulario visual. Cada centroide representa una "palabra visual", formando así el espacio de representación BoVW.

3. Para asignar cada vector SIFT a su palabra visual más cercana (i.e., centroide de K-means), se utilizó `KDTree` de `scikit-learn` en lugar de bibliotecas como `faiss`.

---

### ¿Qué es `KDTree` y por qué se usó?

Un **KDTree (K-dimensional Tree)** es una estructura de datos eficiente para realizar búsquedas de vecinos más cercanos en espacios de múltiples dimensiones. Fue elegida por las siguientes razones:

-  Ofrece **búsqueda rápida de vecinos más cercanos** en espacios moderadamente dimensionales (como los 128 de SIFT, o menos si se reduce con PCA).
-  Ya está implementada y optimizada en `scikit-learn`, lo que evita dependencias externas.
-  Se adapta bien a **asignaciones masivas** como las necesarias en el modelo BoVW, donde se deben procesar millones de vectores SIFT.

> En este proyecto, el árbol KD se construyó con los centroides obtenidos de K-means, y luego se consultó para asignar eficientemente cada vector SIFT a su palabra visual más cercana (operación de clustering inverso).

---

Esta elección técnica permite mantener bajo el tiempo de cómputo incluso al manejar vocabularios visuales grandes (como el de 10,000 palabras), haciendo posible el uso de BoVW en contextos de visión por computadora con miles de imágenes y millones de descriptores.


In [4]:
def entrenar_vocabulario_sift(vectores_sift, n_clusters, batch_size=1000):
    """
    Entrena un modelo MiniBatchKMeans para agrupar vectores SIFT en 'palabras visuales'.

    Parámetros:
    ------------
    vectores_sift : list of np.ndarray
        Lista de matrices SIFT por imagen (cada una de shape N_i × 128).
    
    n_clusters : int
        Número de clústeres deseados para el vocabulario visual (tamaño del vocabulario).
    
    batch_size : int
        Tamaño de los mini-lotes para MiniBatchKMeans (por defecto 1000).

    Retorna:
    --------
    kmeans : MiniBatchKMeans
        Modelo entrenado.
    
    todos_vectores : np.ndarray
        Matriz resultante de apilar todos los vectores SIFT (shape: total_vectores × 128).
    """
    todos_vectores = np.vstack(vectores_sift)  # Une todos los vectores SIFT de todas las imágenes
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=batch_size, verbose=0)  # Inicializa KMeans
    kmeans.fit(todos_vectores)  # Entrena el modelo
    return kmeans, todos_vectores


def construir_indice_kdtree(centroides):
    """
    Construye un índice KDTree con los centroides del vocabulario para búsquedas rápidas.
    
    Parámetros:
    ------------
    centroides : np.ndarray
        Matriz con los centros de los clústeres (palabras visuales).

    Retorna:
    --------
    KDTree : objeto KDTree
        Índice construido para asignación eficiente.
    """
    return KDTree(centroides)


def asignar_palabras_kdtree(index, vectores):
    """
    Asigna cada vector SIFT al centroide más cercano usando KDTree.

    Parámetros:
    ------------
    index : KDTree
        Índice construido sobre los centroides.
    
    vectores : np.ndarray
        Vectores SIFT de una imagen (N × 128).

    Retorna:
    --------
    indices : np.ndarray
        Índices de los centroides asignados (uno por vector).
    """
    _, indices = index.query(vectores, k=1)  # Busca el centroide más cercano
    return indices.flatten()                 # Aplana el resultado para obtener un vector 1D


# Paso 1: Entrenar el vocabulario visual con todos los vectores SIFT
modelo_kmeans, vectores_totales = entrenar_vocabulario_sift(X_train, n_clusters)

# Paso 2: Obtener los centros del modelo y construir el índice KDTree
centros = np.array(modelo_kmeans.cluster_centers_, dtype=np.float32)
index_kdtree = construir_indice_kdtree(centros)

# Paso 3: Asignar los vectores SIFT de una imagen (ej. la primera) a sus palabras visuales
asignaciones = asignar_palabras_kdtree(index_kdtree, X_train[0])

# Mostrar los primeros 10 códigos visuales asignados
print(asignaciones[:10])

[5355 3316 8442 6172 3316 3514 5774 9763 8644 2154]


## Resultado de la asignación de palabras visuales

Después de entrenar el modelo `MiniBatchKMeans` con 10,000 clústeres, se asignaron los vectores SIFT de una imagen a sus respectivos centroides más cercanos utilizando `KDTree`. El resultado fue una lista como la siguiente:

[6942 6605 6659 5973 7247 6556 1008 4761 3597 1219]


## Interpretación

Cada número representa una palabra visual asignada a uno de los vectores SIFT extraídos de la imagen. Es decir, cada vector SIFT fue asignado al centroide más cercano del vocabulario visual aprendido. Por ejemplo:

- El primer vector fue asignado al centroide 6942.
- El segundo al 6605.
- El tercero al 6659.
- Y así sucesivamente.

Este resultado indica qué "palabras visuales" están presentes en la imagen y en qué cantidad. No es una representación final todavía, sino una codificación intermedia que se usará para construir el **histograma de frecuencias** de palabras visuales, también conocido como representación BoVW (*Bag of Visual Words*).

Dicho histograma será una vectorización de la imagen, análoga a cómo se usa la bolsa de palabras en procesamiento de texto, y servirá como entrada para los modelos de clasificación posteriores.


# **Paso 3**: Construcción de histogramas BoVW (Bag of Visual Words)

### Objetivo

Transformar cada imagen en un **vector de características fijo**, utilizando el modelo de *Bag of Visual Words* (BoVW). Esto nos permite representar imágenes —que originalmente tienen una cantidad variable de descriptores SIFT— como vectores de igual longitud, aptos para modelos de clasificación supervisada.

---

### ¿Qué es la matriz `X_train_bovw`?

Es la **matriz de características BoVW**, donde cada fila representa una imagen como un histograma de palabras visuales.

#### Analogía con texto:
Así como en NLP se cuenta cuántas veces aparece cada palabra de un vocabulario en un documento, en visión por computadora se cuenta cuántas veces aparece cada palabra visual (centroide de K-means) en una imagen.

---

### ¿Cómo se construye?

1. A cada imagen se le extraen vectores SIFT.
2. Cada vector se asigna a su centroide más cercano del vocabulario visual (entrenado con K-means).
3. Se cuenta cuántas veces aparece cada centroide en la imagen → eso da lugar a un **histograma**.
4. Se normaliza ese histograma (por ejemplo, con norma L2).
5. Se guarda ese vector como una fila de la matriz `X_train_bovw`.

---

### Dimensiones

Si entrenamos un vocabulario con `n_clusters = 1000` y tenemos 1500 imágenes:

- La matriz resultante será de tamaño:
  
$$ X\_train\_bovw.shape → (1500, 1000) $$


- Cada fila contiene las frecuencias normalizadas de aparición de cada palabra visual en una imagen.

In [5]:
def construir_histogramas_bovw(lista_vectores_sift, kdtree_index, n_clusters):
    """
    Construye la representación Bag of Visual Words (BoVW) para cada imagen a partir de sus vectores SIFT.

    Parámetros:
    ------------
    lista_vectores_sift : list of np.ndarray
        Lista de matrices con los vectores SIFT por imagen (cada matriz: N_i × 128).
    
    kdtree_index : KDTree
        Índice KDTree construido sobre los centroides del vocabulario visual.
    
    n_clusters : int
        Número total de palabras visuales (tamaño del vocabulario, o número de clusters).

    Retorna:
    --------
    X_bovw : np.ndarray
        Matriz de histogramas BoVW, de shape (num_imagenes, n_clusters).
        Cada fila es un vector de frecuencias normalizado (L2) por imagen.
    """

    X_bovw = []  # Lista donde se guardará un histograma por imagen

    for vectores in lista_vectores_sift:
        # Paso 1: Asignar cada vector SIFT al centroide más cercano
        asignaciones = asignar_palabras_kdtree(kdtree_index, vectores)
        
        # Paso 2: Contar cuántos vectores fueron asignados a cada palabra visual (0...n_clusters-1)
        histograma, _ = np.histogram(asignaciones, bins=np.arange(n_clusters + 1))

        # Paso 3: Normalizar el histograma (norma L2) para que la magnitud no dependa del número de keypoints
        histograma = histograma / np.linalg.norm(histograma)

        # Guardar el histograma normalizado
        X_bovw.append(histograma)

    # Convertir lista de histogramas a arreglo numpy (matriz)
    return np.array(X_bovw)


# -----------------------------------------------------
# Construcción del conjunto de entrenamiento BoVW
# -----------------------------------------------------

# Construye los histogramas BoVW para todas las imágenes de entrenamiento
X_train_bovw = construir_histogramas_bovw(X_train, index_kdtree, n_clusters)

# Mostrar tamaño de la matriz resultante y un ejemplo de histograma
print(X_train_bovw.shape)   # Ej. (1500, 10000)
print(X_train_bovw[0])      # Histograma de la primera imagen (frecuencias normalizadas)

(1500, 10000)
[0. 0. 0. ... 0. 0. 0.]


# **PASO 4: Evaluación en el Conjunto de Entrenamiento**

## **VARIANTE 1: Conjunto de entrenamiento sin reducción**

## Clasificación con BoVW

### Objetivo general

Evaluar si la representación Bag of Visual Words (BoVW) permite entrenar modelos de clasificación efectivos para reconocer escenas en imágenes. Para ello, se probaron tres clasificadores distintos:

- **Support Vector Machines (SVM)**
- **Random Forest**
- **Gradient Boosted Trees (GBT)**

Todos los modelos fueron evaluados mediante **validación cruzada de 5 particiones (folds)** sobre los histogramas BoVW previamente generados.

---

## Proceso común a todos los modelos

### **1. Codificación de etiquetas**

Las clases (`'bedroom'`, `'forest'`, `'coast'`, etc.) están en formato de texto. Se utilizó `LabelEncoder` de `scikit-learn` para transformarlas a valores numéricos enteros, compatibles con los clasificadores.

---

### **2. Validación cruzada (`cv=5`)**

Para evaluar el rendimiento de cada modelo se empleó validación cruzada:
- El conjunto de datos se divide en 5 bloques.
- Cada modelo se entrena con 4 bloques y se evalúa en el restante.
- Se repite 5 veces, cambiando el bloque de validación.
- Se calcula el **accuracy promedio** y su **desviación estándar** para medir estabilidad.

---

## Clasificadores evaluados

### Support Vector Machine (SVM)

Modelo entrenado con **kernel RBF (Radial Basis Function)**, que permite capturar relaciones no lineales en los datos.

- `C = 1`: penalización por errores de clasificación.
- `gamma = 'scale'`: ajuste automático del kernel según la varianza de los datos.

### Random Forest

Modelo compuesto por un conjunto de árboles de decisión entrenados con diferentes subconjuntos del conjunto de datos. Es un clasificador robusto que maneja bien datos de alta dimensión y es resistente al sobreajuste.

**Parámetros utilizados:**
- `n_estimators = 100`: número de árboles en el bosque.
- `max_depth = None`: los árboles crecen sin límite de profundidad.
- `random_state = 42`: garantiza reproducibilidad.


### Gradient Boosted Trees (GBT)
Modelo basado en árboles de decisión que se construyen de manera secuencial, donde cada árbol corrige los errores del modelo anterior. Es muy efectivo en problemas complejos con relaciones no lineales.

**Parámetros utilizados:**
- `n_estimators = 100`: número de árboles.
- `learning_rate = 0.1`: controla cuánto contribuye cada nuevo árbol al modelo final.
- `max_depth = 3`: profundidad máxima permitida por cada árbol.
- `random_state = 42`: garantiza reproducibilidad.

In [6]:
# --------------------------------------------
# Codificación de etiquetas de texto a enteros
# --------------------------------------------

le = LabelEncoder()                       # Crea un codificador de etiquetas
y_train_encoded = le.fit_transform(y_train)  # Convierte etiquetas de texto a enteros (necesario para scikit-learn)

# --------------------------------------------
# Definición de clasificadores a evaluar
# --------------------------------------------

clasificadores = {
    "SVM (RBF)": SVC(kernel='rbf', C=1, gamma='scale'),   # SVM con kernel radial, regularización moderada
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42  # Ensamble de 100 árboles sin límite de profundidad
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42  # Árboles pequeños agregados secuencialmente
    )
}

# --------------------------------------------
# Validación cruzada para evaluar clasificadores
# --------------------------------------------

for nombre, modelo in clasificadores.items():
    # Entrena y evalúa el modelo usando validación cruzada de 5 folds
    scores = cross_val_score(modelo, X_train_bovw, y_train_encoded, cv=5)

    # Reporte de desempeño
    print(f"{nombre}")
    print(f"  Accuracy promedio: {scores.mean():.4f}")         # Promedio de accuracy en los 5 folds
    print(f"  Desviación estándar: {scores.std():.4f}\n")     # Desviación estándar para medir estabilidad

SVM (RBF)
  Accuracy promedio: 0.5667
  Desviación estándar: 0.0156

Random Forest
  Accuracy promedio: 0.3827
  Desviación estándar: 0.0139

Gradient Boosting
  Accuracy promedio: 0.2713
  Desviación estándar: 0.0236



## **VARIANTE 2: Evaluación en conjunto de entrenamiento(PCA en vectores SIFT)**

### Objetivo
Reducir la dimensionalidad de los vectores SIFT (de 128 dimensiones) **antes** de construir el vocabulario visual mediante K-means. Esto permite disminuir el costo computacional del agrupamiento y posiblemente eliminar redundancia en las descripciones locales.

---

### Paso 1: Aplicar PCA a todos los vectores SIFT del conjunto de entrenamiento

- Se apilaron todos los vectores SIFT en una sola matriz.
- Se aplicó **PCA** para reducir a `n_components = 50`.

---

### Paso 2: Reconstruir la lista de vectores por imagen

- Se volvió a separar la matriz reducida en una lista por imagen, como en la estructura original `X_train`.

---

### Paso 3: Entrenar K-means y construir el índice KDTree

- Se entrenó `MiniBatchKMeans` con los vectores ya reducidos.
- Se construyó un `KDTree` con los centroides obtenidos.

---

### Paso 4: Construcción de histogramas BoVW con vectores reducidos

- Se generó un histograma normalizado para cada imagen utilizando los vectores reducidos y el vocabulario aprendido.

---

### Paso 5: Clasificación con validación cruzada

- Se evaluaron tres clasificadores (SVM, RF y GBT) sobre los histogramas BoVW obtenidos con reducción de dimensión.


In [7]:
# --------------------------------------------------------------------------
# PASO 1: Reducción de dimensionalidad en vectores SIFT
# --------------------------------------------------------------------------

# Número de componentes para reducción (de 128 → 50)
n_pca_sift = 50

# Combina todos los vectores SIFT en una sola matriz (shape: total_vectores × 128)
vectores_sift_apilados = np.vstack(X_train)

# Ajusta PCA y transforma todos los vectores
pca_sift = PCA(n_components=n_pca_sift)
vectores_sift_reducidos = pca_sift.fit_transform(vectores_sift_apilados)

# --------------------------------------------------------------------------
# PASO 2: Reconstrucción por imagen
# --------------------------------------------------------------------------

# Se separan nuevamente los vectores por imagen, respetando su tamaño original
X_train_pca = []
i = 0
for vectores in X_train:
    n = vectores.shape[0]                          # cuántos keypoints tenía esta imagen
    X_train_pca.append(vectores_sift_reducidos[i:i+n])  # se toman esos n vectores reducidos
    i += n

# --------------------------------------------------------------------------
# PASO 3: Vocabulario visual con vectores reducidos
# --------------------------------------------------------------------------

# Se entrena el modelo MiniBatchKMeans con vectores SIFT reducidos
modelo_kmeans_pca, _ = entrenar_vocabulario_sift(X_train_pca, n_clusters)

# Se obtienen los centroides del vocabulario (palabras visuales) y se construye el KDTree
centros_pca = modelo_kmeans_pca.cluster_centers_
index_kdtree_pca = construir_indice_kdtree(centros_pca)

# --------------------------------------------------------------------------
# PASO 4: Construcción de histogramas BoVW con vectores reducidos
# --------------------------------------------------------------------------

# Se generan los histogramas BoVW a partir de los vectores SIFT reducidos
X_train_bovw_pca = construir_histogramas_bovw(X_train_pca, index_kdtree_pca, n_clusters)

# --------------------------------------------------------------------------
# PASO 5: Validación cruzada con los histogramas BoVW reducidos
# --------------------------------------------------------------------------

# Se reutiliza el mismo vector de etiquetas codificadas (y_train_encoded)
for nombre, modelo in clasificadores.items():
    scores = cross_val_score(modelo, X_train_bovw_pca, y_train_encoded, cv=5)

    # Mostrar resultados para cada clasificador con PCA aplicado a SIFT
    print(f"PCA-SIFT | {nombre}")
    print(f"  Accuracy promedio: {scores.mean():.4f}")
    print(f"  Desviación estándar: {scores.std():.4f}\n")

PCA-SIFT | SVM (RBF)
  Accuracy promedio: 0.5647
  Desviación estándar: 0.0175

PCA-SIFT | Random Forest
  Accuracy promedio: 0.3653
  Desviación estándar: 0.0145

PCA-SIFT | Gradient Boosting
  Accuracy promedio: 0.2647
  Desviación estándar: 0.0235



# **VARIANTE 3: Reducción en histogramas BoVW antes de la clasificación**

###  Objetivo
Aplicar **PCA** directamente sobre los histogramas BoVW (Bag of Visual Words), para reducir su dimensión antes de entrenar los clasificadores. Esto permite reducir el tamaño de entrada al clasificador y posiblemente mejorar el rendimiento eliminando ruido.

---

### Paso 1: Aplicar PCA a los histogramas `X_train_bovw`

---

### Paso 2: Clasificación con histogramas reducidos

- Se evalúan los clasificadores (SVM, RF, GBT) con validación cruzada (`cv=5`) sobre los histogramas reducidos.


In [8]:
# --------------------------------------------------------------------------
# PASO 1: Reducción de dimensionalidad en los histogramas BoVW
# --------------------------------------------------------------------------

# Número de componentes PCA a conservar (de 10,000 → 300)
n_pca_bovw = 300  # Se eligió este valor por retener >95% de la varianza y reducir la carga computacional

# Ajusta PCA a la matriz de histogramas BoVW y transforma los datos
pca_bovw = PCA(n_components=n_pca_bovw)
X_train_bovw_reducido = pca_bovw.fit_transform(X_train_bovw)

# Resultado: nueva matriz de entrenamiento (shape: num_imagenes × 300)

# --------------------------------------------------------------------------
# PASO 2: Evaluación de clasificadores con BoVW reducidos
# --------------------------------------------------------------------------

for nombre, modelo in clasificadores.items():
    # Validación cruzada de 5 folds con los histogramas reducidos
    scores = cross_val_score(modelo, X_train_bovw_reducido, y_train_encoded, cv=5)

    # Resultados del clasificador
    print(f" PCA-BoVW | {nombre}")
    print(f" Accuracy promedio: {scores.mean():.4f}")
    print(f" Desviación estándar: {scores.std():.4f}\n")

 PCA-BoVW | SVM (RBF)
 Accuracy promedio: 0.5787
 Desviación estándar: 0.0120

 PCA-BoVW | Random Forest
 Accuracy promedio: 0.4613
 Desviación estándar: 0.0252

 PCA-BoVW | Gradient Boosting
 Accuracy promedio: 0.4407
 Desviación estándar: 0.0299



# **VARIANTE 4: Reducción en vectores SIFT y en histogramas BoVW**

### Objetivo

Aplicar reducción de dimensionalidad con PCA en dos etapas del pipeline:
1. Sobre los vectores SIFT (antes de K-means).
2. Sobre los histogramas BoVW (antes de la clasificación).

Este enfoque combina los beneficios de ambas variantes anteriores, reduciendo tanto la complejidad del vocabulario visual como la entrada final a los clasificadores.

---

### Paso 1: Aplicar PCA a los vectores SIFT

- Se apilaron todos los vectores SIFT.
- Se redujeron de 128 a `n_pca_sift = 50` componentes.
- Luego se reconstruyó la lista por imagen para continuar el pipeline.

---

### Paso 2: Construcción del vocabulario visual y BoVW con vectores reducidos

- Se entrenó MiniBatchKMeans sobre los vectores SIFT reducidos.
- Se construyó el índice KDTree y los histogramas BoVW.

---

### Paso 3: Aplicar PCA a los histogramas BoVW

- Se redujo el número de dimensiones de los histogramas.
- Para pruebas, se usaron pocos componentes (`n_pca_bovw = 8`).

---

### Paso 4: Clasificación con validación cruzada

- Se evaluaron los tres clasificadores sobre los histogramas reducidos.

---

In [9]:
# --------------------------------------------
# PASO 1: Aplicar PCA a vectores SIFT
# --------------------------------------------

n_pca_sift = 50  # Reducir los vectores SIFT de 128 a 50 dimensiones

# Combina todos los vectores SIFT en una sola matriz (shape: total_vectores × 128)
vectores_sift_apilados = np.vstack(X_train)

# Aplica PCA para reducir dimensionalidad
pca_sift = PCA(n_components=n_pca_sift)
vectores_sift_reducidos = pca_sift.fit_transform(vectores_sift_apilados)

# Reconstruir la lista por imagen, manteniendo la estructura original
X_train_pca = []
i = 0
for vectores in X_train:
    n = vectores.shape[0]
    X_train_pca.append(vectores_sift_reducidos[i:i+n])
    i += n

# --------------------------------------------
# PASO 2: Entrenar vocabulario visual y construir histogramas BoVW
# --------------------------------------------

# Se entrena el modelo MiniBatchKMeans con los vectores SIFT reducidos
modelo_kmeans_pca, _ = entrenar_vocabulario_sift(X_train_pca, n_clusters)

# Construye índice KDTree con los centroides aprendidos
centros_pca = modelo_kmeans_pca.cluster_centers_
index_kdtree_pca = construir_indice_kdtree(centros_pca)

# Construye histogramas BoVW con los vectores SIFT reducidos
X_train_bovw_pca = construir_histogramas_bovw(X_train_pca, index_kdtree_pca, n_clusters)

# --------------------------------------------
# PASO 3: Aplicar PCA a los histogramas BoVW
# --------------------------------------------

n_pca_bovw = 300  # Reducir histogramas de BoVW de 10,000 a 300 componentes

# Ajusta PCA a los histogramas y transforma
pca_bovw = PCA(n_components=n_pca_bovw)
X_train_bovw_doble_pca = pca_bovw.fit_transform(X_train_bovw_pca)

# --------------------------------------------
# PASO 4: Evaluación con validación cruzada
# --------------------------------------------

print(f"\n PCA en SIFT + BoVW ({n_pca_sift} dims SIFT → KMeans + {n_pca_bovw} dims BoVW → clasificador)\n")

# Evalúa los clasificadores con validación cruzada de 5 folds
for nombre, modelo in clasificadores.items():
    scores = cross_val_score(modelo, X_train_bovw_doble_pca, y_train_encoded, cv=5)

    print(f" {nombre}")
    print(f" Accuracy promedio: {scores.mean():.4f}")
    print(f" Desviación estándar: {scores.std():.4f}\n")


 PCA en SIFT + BoVW (50 dims SIFT → KMeans + 300 dims BoVW → clasificador)

 SVM (RBF)
 Accuracy promedio: 0.5793
 Desviación estándar: 0.0139

 Random Forest
 Accuracy promedio: 0.4713
 Desviación estándar: 0.0109

 Gradient Boosting
 Accuracy promedio: 0.4387
 Desviación estándar: 0.0245



## Comparación de Accuracy promedio en el conjunto de entrenamiento

| Clasificador           | Variante 1 (Sin reducción) | Variante 2 (PCA en SIFT) | Variante 3 (PCA en BoVW) | Variante 4 (PCA en SIFT + BoVW) |
|------------------------|----------------------------|--------------------------|--------------------------|----------------------------------|
| **SVM (RBF)**          | 0.5667 ± 0.0156            | 0.5647 ± 0.0175          | **0.5787 ± 0.0120**      | 0.5793 ± 0.0139                  |
| **Random Forest**      | 0.3827 ± 0.0139            | 0.3653 ± 0.0145          | **0.4613 ± 0.0252**      | 0.4713 ± 0.0109                  |
| **Gradient Boosting**  | 0.2713 ± 0.0236            | 0.2647 ± 0.0235          | **0.4407 ± 0.0299**      | 0.4387 ± 0.0245                  |

---

**Observaciones:**
- La **Variante 3** (PCA en BoVW) obtuvo el mejor rendimiento general en los tres clasificadores.
- La reducción en vectores SIFT (Variante 2) tuvo un impacto leve o negativo en comparación con la versión sin reducción.
- La combinación completa (Variante 4) logró buenos resultados, especialmente en RF y GBT, aunque no superó a la reducción solo en BoVW.

# **PASO 5: Evaluación en el Conjunto de Validación**

## **VARIANTE 1: Conjunto de prueba sin reducción**

### Objetivo
Evaluar el rendimiento de tres clasificadores utilizando la representación BoVW construida directamente a partir de los vectores SIFT **sin aplicar reducción de dimensionalidad**.

---

### Paso 1: Procesamiento del conjunto de entrenamiento

- Se cargaron los vectores SIFT desde los archivos `.npy`.
- Se entrenó un modelo de K-means con `n_clusters = 10,000`.
- Se construyó el índice KDTree para asignar palabras visuales.
- Se generaron los histogramas BoVW de tamaño igual al número de clusters.

---

### Paso 2: Procesamiento del conjunto de prueba

---

### Paso 3: Entrenamiento y evaluación en conjunto de prueba


In [10]:
# --------------------------------------------
# PASO 1: Cargar vectores SIFT del conjunto de prueba
# --------------------------------------------

ruta_test = "./imagedb/data_tarea/test"  # Ruta donde están las carpetas de test por clase
X_test, y_test, _, rutas_test = cargar_vectores_sift(ruta_test)  # Carga vectores y etiquetas

# --------------------------------------------
# PASO 2: Construir histogramas BoVW con KDTree original
# --------------------------------------------

# Se usa el KDTree entrenado con los centroides del conjunto de entrenamiento
# Esto asegura que las "palabras visuales" sean consistentes entre entrenamiento y prueba
X_test_bovw = construir_histogramas_bovw(X_test, index_kdtree, n_clusters)

# --------------------------------------------
# PASO 3: Codificar etiquetas del conjunto de prueba
# --------------------------------------------

# Usa el mismo LabelEncoder que se usó para el conjunto de entrenamiento
y_test_encoded = le.transform(y_test)

# --------------------------------------------
# PASO 4: Entrenar clasificadores con X_train_bovw y evaluar en X_test_bovw
# --------------------------------------------

for nombre, modelo in clasificadores.items():
    modelo.fit(X_train_bovw, y_train_encoded)  # Entrena con el conjunto de entrenamiento completo
    y_pred = modelo.predict(X_test_bovw)       # Predice clases para las imágenes de prueba

    # Mostrar métricas de desempeño
    print("-"*100)
    print(f"\n Resultados en conjunto de prueba — {nombre}")
    print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")  # Accuracy global
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_encoded, y_pred))                   # Confusión por clase

    print("\nReporte de clasificación:")
    print(classification_report(y_test_encoded, y_pred, target_names=categorias))  # Precision, recall, f1

----------------------------------------------------------------------------------------------------

 Resultados en conjunto de prueba — SVM (RBF)
Accuracy: 0.5645
Matriz de confusión:
[[ 44   2   1   5   4   1  23   7   3   5   3   3   9   0   6]
 [  8 169   6  28   6   0   0   0   8   1  30   0   1   2   1]
 [  0   0 201   0   0   0   0   0  11   0   5   2   0   9   0]
 [  3  12   2 116  11   0   1   0   1   0   7   0   4   1   2]
 [ 26   4   1   3  61  15  14  11   5   5   2  33  19   8   4]
 [  6   0   0   3  14 114  15   1   0   5   1  19  17   4   9]
 [ 16   0   0   0   3   9  61   5   0  13   0   3   0   0   0]
 [ 57   1   0   1  15   2  31  49   0  16   1   9   2   4   1]
 [ 15  18  17  13   1   0   0   0 163   0  20   0   6  21   0]
 [  4   0   0   0   1   0  17   3   0  90   0   0   0   0   0]
 [  0  44  43  18   3   0   0   1  57   0 121   1   2  19   1]
 [  6   0   3   0  12  18  11   6   3   4   0 129  12   5   6]
 [  1   0   0   2  24  13   3   1   9   0   9  17 103   5 

## **VARIANTE 2: Evaluación en conjunto de prueba (PCA en vectores SIFT)**

### Objetivo
Evaluar el desempeño de los clasificadores cuando los vectores SIFT del conjunto de prueba se reducen mediante PCA (entrenado previamente con `X_train`), y luego se procesan mediante el vocabulario visual aprendido con los vectores SIFT reducidos.

---

### Observaciones
Esta variante permite evaluar si la reducción previa en vectores SIFT mejora o empeora el rendimiento al clasificar imágenes nuevas. La comparación con la Variante 1 (sin reducción) ayuda a valorar el impacto de eliminar redundancia desde la etapa de descriptores locales.


In [11]:
# --------------------------------------------
# PASO 1: Cargar vectores SIFT originales del conjunto de prueba
# --------------------------------------------

ruta_test = "./imagedb/data_tarea/test"
X_test, y_test, _, rutas_test = cargar_vectores_sift(ruta_test)

# Se cargan los vectores SIFT sin reducir, tal como se hizo con X_train inicialmente

# --------------------------------------------
# PASO 2: Aplicar PCA (entrenado con X_train) a vectores SIFT de prueba
# --------------------------------------------

X_test_pca = []
for vectores in X_test:
    vectores_pca = pca_sift.transform(vectores)  # Se transforma cada imagen usando el PCA entrenado con entrenamiento
    X_test_pca.append(vectores_pca)

# Ahora X_test_pca contiene vectores SIFT reducidos a 50 dimensiones por imagen

# --------------------------------------------
# PASO 3: Construir histogramas BoVW del conjunto de prueba
# --------------------------------------------

# Se usan las mismas "palabras visuales" (KDTree) construidas con los vectores SIFT reducidos del entrenamiento
X_test_bovw_pca = construir_histogramas_bovw(X_test_pca, index_kdtree_pca, n_clusters)

# --------------------------------------------
# PASO 4: Codificar etiquetas del conjunto de prueba
# --------------------------------------------

y_test_encoded = le.transform(y_test)  # Usa el codificador de etiquetas del entrenamiento

# --------------------------------------------
# PASO 5: Evaluar clasificadores entrenados con BoVW reducidos
# --------------------------------------------

print("Resultados en conjunto de prueba — Variante 2 (PCA en SIFT)\n")

for nombre, modelo in clasificadores.items():
    modelo.fit(X_train_bovw_pca, y_train_encoded)     # Entrena sobre el BoVW reducido del conjunto de entrenamiento
    y_pred = modelo.predict(X_test_bovw_pca)          # Predice etiquetas para el conjunto de prueba

    # Reporte de métricas
    print("-"*100)
    print(f"\n Resultados en conjunto de prueba — {nombre}")
    print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")           # Precisión total
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_encoded, y_pred))                            # Errores por clase
    print("\nReporte de clasificación:")
    print(classification_report(y_test_encoded, y_pred, target_names=categorias))  # Métricas detalladas por clase

Resultados en conjunto de prueba — Variante 2 (PCA en SIFT)

----------------------------------------------------------------------------------------------------

 Resultados en conjunto de prueba — SVM (RBF)
Accuracy: 0.5682
Matriz de confusión:
[[ 56   1   1   6   4   2  14   5   3   7   3   2   7   0   5]
 [  5 176   5  26   2   0   0   1   7   1  32   0   1   3   1]
 [  0   0 201   0   0   0   0   0  10   0   7   0   1   9   0]
 [  3  13   2 110  12   0   0   0   2   0   9   2   4   1   2]
 [ 24   6   1   5  58  22  10  13   5   6   3  35  12   6   5]
 [  6   1   0   2  13 123  14   1   0   3   1  21  13   2   8]
 [ 22   0   0   0   6   7  50   2   0  20   0   2   0   0   1]
 [ 65   1   0   1   8   3  25  42   0  21   0  13   4   5   1]
 [ 12  20  15  12   1   0   0   0 172   0  15   0   6  21   0]
 [  8   0   0   0   0   0  12   4   0  89   0   2   0   0   0]
 [  1  39  43  21   1   0   0   1  61   0 125   0   2  15   1]
 [  8   0   2   0  15  14  14   1   2   5   1 127  14   5   

## **Variante 3: Evaluación en conjunto de prueba (PCA en histogramas BoVW)**

### Objetivo
Evaluar el desempeño de los clasificadores cuando se reduce la dimensionalidad de los histogramas BoVW mediante PCA **antes de entrenar y clasificar**, manteniendo los vectores SIFT originales sin reducción.

---

### Observaciones
En esta variante se evaluó si reducir la dimensionalidad de los histogramas de características (BoVW) puede conservar la información relevante para clasificación, al tiempo que reduce la complejidad computacional y el riesgo de sobreajuste.

In [12]:
# --------------------------------------------
# PASO 1: Cargar vectores SIFT del conjunto de prueba
# --------------------------------------------

ruta_test = "./imagedb/data_tarea/test"
X_test, y_test, _, rutas_test = cargar_vectores_sift(ruta_test)

# Se cargan los vectores SIFT sin reducción, igual que en entrenamiento

# --------------------------------------------
# PASO 2: Construir histogramas BoVW (sin PCA en SIFT)
# --------------------------------------------

# Se construyen histogramas con el KDTree entrenado originalmente sobre vectores SIFT completos
X_test_bovw = construir_histogramas_bovw(X_test, index_kdtree, n_clusters)

# --------------------------------------------
# PASO 3: Aplicar el mismo PCA (con 300 componentes) al conjunto de prueba
# --------------------------------------------

# Se transforma X_test_bovw usando el PCA entrenado previamente sobre X_train_bovw
X_test_bovw_reducido = pca_bovw.transform(X_test_bovw)

# --------------------------------------------
# PASO 4: Codificar etiquetas del conjunto de prueba
# --------------------------------------------

y_test_encoded = le.transform(y_test)  # Se usa el mismo codificador del entrenamiento

# --------------------------------------------
# PASO 5: Evaluar clasificadores entrenados con X_train_bovw_reducido
# --------------------------------------------

print("Resultados en conjunto de prueba — Variante 3 (PCA en BoVW)\n")

for nombre, modelo in clasificadores.items():
    modelo.fit(X_train_bovw_reducido, y_train_encoded)      # Entrenamiento con datos reducidos
    y_pred = modelo.predict(X_test_bovw_reducido)           # Evaluación en prueba (también reducida)

    # Mostrar resultados de evaluación
    print("-"*100)
    print(f"\n Resultados en conjunto de prueba — {nombre}")
    print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_encoded, y_pred))
    print("\nReporte de clasificación:")
    print(classification_report(y_test_encoded, y_pred, target_names=categorias))


Resultados en conjunto de prueba — Variante 3 (PCA en BoVW)

----------------------------------------------------------------------------------------------------

 Resultados en conjunto de prueba — SVM (RBF)
Accuracy: 0.0667
Matriz de confusión:
[[  0   0   0   0   0   0   0   0   3   0   0  90   0  23   0]
 [  0   0   1   0   0   0   0   0  17   0   0 183   0  59   0]
 [  0   0   0   0   0   0   0   0   3   0   0 190   1  34   0]
 [  0   0   1   0   0   0   0   0   5   0   0 117   0  37   0]
 [  0   0   0   0   0   0   0   0   4   0   0 158   1  48   0]
 [  0   0   0   0   0   0   0   0   6   0   0 146   0  56   0]
 [  0   0   0   0   0   0   0   0   6   0   0  81   1  22   0]
 [  0   0   0   0   0   0   0   0   3   0   0 137   0  49   0]
 [  0   0   0   0   0   0   0   0   2   0   0 206   0  66   0]
 [  0   0   0   0   0   0   0   0  10   0   0  75   0  30   0]
 [  0   0   0   0   0   0   0   0   4   0   0 247   1  58   0]
 [  0   0   0   0   0   0   0   0   2   0   0 169   0  44   

/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

----------------------------------------------------------------------------------------------------

 Resultados en conjunto de prueba — Random Forest
Accuracy: 0.0707
Matriz de confusión:
[[  0   0   3   1  10   0  11  11  24   0   8  25  22   1   0]
 [  0   0   2   0  21   2  23  16  64   3  13  67  41   6   2]
 [  0   0   0   0   9   1  17  24  45   0   4  81  41   6   0]
 [  1   0   2   1  10   2   9  10  32   4   5  48  32   4   0]
 [  0   0   1   2  11   1  14  22  41   2   7  57  43  10   0]
 [  0   0   0   0  15   1  11  11  44   5   8  56  40  17   0]
 [  0   0   0   1   5   0   4   6  29   3   8  29  24   1   0]
 [  0   0   0   0   7   0  13  22  37   1   1  66  27  15   0]
 [  0   0   4   1  18   1  12  29  47   0   8 103  42   9   0]
 [  0   0   1   3   9   0   9   6  33   2   5  23  20   4   0]
 [  0   0   4   4  18   1  24  17  55   1   6 116  50  14   0]
 [  2   0   0   0  10   1  17  25  40   0   1  76  31  12   0]
 [  0   0   3   1  12   0   9  23  28   2   3  74  30 

/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/danirm/anaconda3/envs/aprendizaje_automatizado/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

## **VARIANTE 4: Evaluación en conjunto de prueba (PCA en vectores SIFT y en histogramas BoVW)**

### Objetivo
Evaluar el rendimiento de los clasificadores cuando se aplica reducción de dimensionalidad tanto:
- A los vectores SIFT antes del K-means (como en la Variante 2).
- A los histogramas BoVW antes de la clasificación (como en la Variante 3).

Esta combinación busca compactar la información desde los descriptores locales hasta la representación final usada por el clasificador.

---

### Observaciones
Esta última variante permite evaluar si la doble reducción de dimensionalidad ayuda a obtener modelos más ligeros sin sacrificar (o incluso mejorando) el rendimiento en clasificación. Es útil especialmente cuando se trabaja con vocabularios visuales grandes y descriptores densos.

In [13]:
# --------------------------------------------
# PASO 1: Cargar vectores SIFT del conjunto de prueba
# --------------------------------------------

ruta_test = "./imagedb/data_tarea/test"
X_test, y_test, _, rutas_test = cargar_vectores_sift(ruta_test)

# Carga los vectores SIFT originales del conjunto de prueba, sin reducción aún.

# --------------------------------------------
# PASO 2: Aplicar PCA entrenado a vectores SIFT de prueba
# --------------------------------------------

X_test_pca = []
for vectores in X_test:
    vectores_reducidos = pca_sift.transform(vectores)  # Usa el PCA entrenado con vectores SIFT de entrenamiento
    X_test_pca.append(vectores_reducidos)

# Ahora cada imagen del conjunto de prueba tiene vectores SIFT reducidos (e.g., 50 dimensiones).

# --------------------------------------------
# PASO 3: Construir histogramas BoVW con KDTree entrenado con SIFT reducidos
# --------------------------------------------

# Se construyen histogramas BoVW con las palabras visuales aprendidas con vectores SIFT reducidos (index_kdtree_pca)
X_test_bovw_pca = construir_histogramas_bovw(X_test_pca, index_kdtree_pca, n_clusters)

# --------------------------------------------
# PASO 4: Aplicar PCA a histogramas BoVW del conjunto de prueba
# --------------------------------------------

# Se aplica el PCA entrenado con los histogramas del conjunto de entrenamiento
X_test_bovw_doble_pca = pca_bovw.transform(X_test_bovw_pca)

# --------------------------------------------
# PASO 5: Codificar etiquetas del conjunto de prueba
# --------------------------------------------

y_test_encoded = le.transform(y_test)  # Codifica etiquetas usando el mismo LabelEncoder

# --------------------------------------------
# PASO 6: Evaluar clasificadores entrenados con doble reducción
# --------------------------------------------

print("Resultados en conjunto de prueba — Variante 4 (PCA en SIFT y BoVW)\n")

for nombre, modelo in clasificadores.items():
    modelo.fit(X_train_bovw_doble_pca, y_train_encoded)      # Entrena con el conjunto reducido
    y_pred = modelo.predict(X_test_bovw_doble_pca)           # Evalúa también con doble reducción

    # Mostrar resultados
    print("-"*100)
    print(f"\n Resultados en conjunto de prueba — {nombre}")
    print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_encoded, y_pred))
    print("\nReporte de clasificación:")
    print(classification_report(y_test_encoded, y_pred, target_names=categorias))

Resultados en conjunto de prueba — Variante 4 (PCA en SIFT y BoVW)

----------------------------------------------------------------------------------------------------

 Resultados en conjunto de prueba — SVM (RBF)
Accuracy: 0.4630
Matriz de confusión:
[[ 46   1   3   1   1   0   1  11   2  17   1  14   6  11   1]
 [  7 129   9  24   2   0   0   0  21   2  56   0   2   8   0]
 [  0   0 223   0   0   0   0   0   0   0   0   0   0   5   0]
 [  4   4   6  78   1   0   0   1   7   0  13   4  12  29   1]
 [ 22   2   5   4   8   7   0  11   2  14   3  75  15  42   1]
 [  9   0   2   1   5  60   3   1   1  18   0  76  14  15   3]
 [ 11   0   0   0   3   2   1   1   0  69   0  20   0   3   0]
 [ 33   1   0   0   0   1   2  43   1  43   0  40   2  22   1]
 [  6   9  59   5   0   0   0   0 133   0   8   0   1  53   0]
 [  2   0   0   0   0   0   0   1   0 105   0   6   0   1   0]
 [  3  11  90   6   0   0   0   0  69   0  96   0   5  29   1]
 [  3   0  10   0   1   0   1   0   0  14   0 165   6

## Accuracy en conjunto de prueba por variante y clasificador

| Clasificador           | Variante 1 (Sin reducción) | Variante 2 (PCA en SIFT) | Variante 3 (PCA en BoVW) | Variante 4 (PCA en SIFT + BoVW) |
|------------------------|----------------------------|--------------------------|--------------------------|----------------------------------|
| **SVM (RBF)**          | **0.5645**                 | 0.5682                   | 0.0667                   | 0.4630                           |
| **Random Forest**      | 0.3678                     | 0.3873                  | 0.0707                   | 0.4868                           |
| **Gradient Boosting**  | 0.2720                     | 0.2777                   | 0.0650                   | **0.5075**                       |

---

###  Observaciones:

- **SVM (RBF)** obtuvo su mejor desempeño en la **Variante 1**, con una ligera caída en la Variante 2. La **Variante 3** colapsó completamente, lo cual sugiere que la reducción PCA en BoVW eliminó demasiada información discriminativa.
  
- **Random Forest** mostró un bajo rendimiento en general, pero mejoró significativamente en la **Variante 4**, posiblemente por la combinación equilibrada entre reducción en SIFT y BoVW.

- **Gradient Boosting** fue el gran beneficiado de la **Variante 4**, alcanzando su mayor precisión (0.5055). Esto indica que este clasificador logra aprovechar mejor la información comprimida de ambas reducciones.

- En la **Variante 3**, todos los clasificadores sufrieron una caída severa de rendimiento, lo cual sugiere que **hacer reducción únicamente en BoVW sin procesar los SIFT** puede no ser recomendable para este conjunto de datos.

# **Conclusiones Generales**

## Comparación de Accuracy promedio

### Conjunto de entrenamiento

| Clasificador           | Variante 1 (Sin reducción) | Variante 2 (PCA en SIFT) | Variante 3 (PCA en BoVW) | Variante 4 (PCA en SIFT + BoVW) |
|------------------------|----------------------------|--------------------------|--------------------------|----------------------------------|
| **SVM (RBF)**          | 0.5667 ± 0.0156            | 0.5647 ± 0.0175          | **0.5787 ± 0.0120**      | 0.5793 ± 0.0139                  |
| **Random Forest**      | 0.3827 ± 0.0139            | 0.3653 ± 0.0145          | **0.4613 ± 0.0252**      | 0.4713 ± 0.0109                  |
| **Gradient Boosting**  | 0.2713 ± 0.0236            | 0.2647 ± 0.0235          | **0.4407 ± 0.0299**      | 0.4387 ± 0.0245                  |

---

### Conjunto de prueba

| Clasificador           | Variante 1 (Sin reducción) | Variante 2 (PCA en SIFT) | Variante 3 (PCA en BoVW) | Variante 4 (PCA en SIFT + BoVW) |
|------------------------|----------------------------|--------------------------|--------------------------|----------------------------------|
| **SVM (RBF)**          | **0.5645**                 | 0.5682                   | 0.0667                   | 0.4630                           |
| **Random Forest**      | 0.3678                     | 0.3873                  | 0.0707                   | 0.4868                           |
| **Gradient Boosting**  | 0.2720                     | 0.2777                   | 0.0650                   | **0.5075**                       |
---

- A pesar de que **SVM (RBF)** fue consistentemente el modelo más preciso, su desempeño cayó drásticamente en la Variante 3, lo que indica que **reducir directamente los histogramas BoVW sin procesar previamente los vectores SIFT puede no ser una buena idea**.
  
- **Random Forest** mostró una mejora notable en la Variante 4, lo que sugiere que **la combinación de reducciones (en SIFT y BoVW)** permite simplificar el espacio de características sin perder información relevante.

- **Gradient Boosting** fue el único modelo que alcanzó su mejor rendimiento en la **Variante 4**, obteniendo un accuracy de **0.5055** en prueba. Esto refuerza que este tipo de modelos pueden **beneficiarse de representaciones comprimidas y más estructuradas**.

- **La Variante 3 fue la menos efectiva** para todos los clasificadores, lo cual indica que aplicar PCA únicamente sobre los histogramas BoVW, sin reducir primero los vectores SIFT, puede eliminar información importante.

---

## Respuestas Teóricas

### 1. Justificación del número de componentes en PCA

Se eligieron **50 componentes** para la reducción de vectores SIFT y **300 componentes** para la reducción de histogramas BoVW. Esta elección se basó en:

- La varianza acumulada explicada por los primeros componentes (alrededor del 90–95% para 50 en SIFT y >95% para 300 en BoVW).
- Consideraciones prácticas de eficiencia computacional, debido a que se trabaja con 10,000 clústeres visuales.
- Experimentos preliminares que mostraron estabilidad en el rendimiento a partir de esos umbrales.

---

### 2. Ventajas y desventajas de los modelos utilizados

#### **SVM (RBF)**:
- Alta precisión en la mayoría de los casos.
- Tiempo de entrenamiento más alto, especialmente con muchos datos.

#### **Random Forest**:
- Entrenamiento rápido y robusto al sobreajuste.
- Menor precisión con datos de alta dimensionalidad sin reducción previa.

#### **Gradient Boosted Trees**:
- Captura relaciones complejas y suele generalizar bien.
- Sensible a hiperparámetros, más lento y menos estable si no se afina.

---

### 3. Analogía de *stop word* en clasificación de imágenes

En procesamiento de lenguaje natural (NLP), una **stop word** es una palabra frecuente y poco informativa (como "el", "de", "que"). En el contexto de imágenes, el equivalente podría ser:

> **Palabras visuales frecuentes pero no discriminativas**, como bordes comunes, texturas repetitivas o patrones que aparecen en todas las clases sin aportar información útil.

Una posible estrategia sería:
- Eliminar los clústeres más frecuentes del vocabulario BoVW.
- Asignarles menor peso durante la construcción del histograma (ej. TF-IDF visual).

Esto podría ayudar a enfocar el modelo en características más distintivas y mejorar el desempeño.